# Inteligentne Obliczenia projekt Przewidywanie skuteczności kampanii promocyjnej

## Etap I - Przygotowania i obróbka danych

W pierwszym kroku odbywa się import wszystkich bibliotek, które będą potrzebne w kolejnych etapach projektu.

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from data_preparation import generate_sets

# Wstępna obróbka danych przed pracą z danymi ciągłymi
Przed zastosowaniem jakichkolwiek technik analizy danych, pierwotny zestaw danych wymagał wstępnego przygotowania. Modele uczenia maszynowego, gdy w projekcie wykorzystujemy bibliotekę scikit-learn, wymagają danych w formacie czysto numerycznym, bez wartości pustych. Poniżej znajduje się opis przygotowania danych:

1. Detekcja i obsługa braków danych oraz cech bezużytecznych

- Wypełnienie braków w zmiennej Income: Zauwazono występowanie braków danych (NaN) w kolumnie reprezentującej dochód gospodarstwa domowego. Ponieważ dochód może zawierać wartości skrajne, zastąpienie braków średnią arytmetyczną mogłoby zaburzyć rozkład. Zdecydowano się na użycie mediany.

- Usunięcie cech o zerowej wariancji: Zidentyfikowano kolumny Z_CostContact oraz Z_Revenue, które dla każdej obserwacji w zbiorze przyjmowały dokładnie taką samą wartość. Z punktu widzenia budowy modelu klasyfikacyjnego takie cechy nie wnoszą zadnego rozróznienia danych, dlatego zostały trwale usunięte.

2. Inżynieria cech

- Dane dotyczące dat są słabo czytelne dla standardowych algorytmów klasyfikacyjnych. Konieczne było wyciągnięcie z nich wartości numerycznych, które realnie opisują zachowanie klienta:

- Wiek klienta: Zamiast operować rokiem urodzenia, stworzono nową zmienną reprezentującą fizyczny wiek. Dodatkowo odfiltrowano rekordy niemożliwe z logicznego punktu widzenia (np. osoby w wieku powyżej 100 lat), które mogły wynikać z błędów przy wprowadzaniu danych.

- Czas w systemie: Datę pierwszej rejestracji klienta (Dt_Customer) przekształcono na liczbę dni, jaka upłynęła od jego dołączenia. Pozwoliło to uzyskać ciągłą zmienną numeryczną, która może być wskaźnikiem lojalności klienta.

3. Kodowanie zmiennych kategorycznych

- Kolumny tekstowe, takie jak Education czy Marital_Status, nie mogą zostać bezpośrednio przetworzone przez modele matematyczne. Zastosowano technikę wskaźnikową z wykorzystaniem funkcji pd.get_dummies. Aby uniknąć problemu współliniowości (tzw. dummy variable trap, gdzie jedna zmienna jest idealną kombinacją liniową pozostałych), zastosowano parametr drop_first=True.

4. Separacja zbiorów

- Ostatnim etapem obróbki był podział danych na zbiór treningowy (80%) i testowy (20%). Ponieważ w kampaniach marketingowych odsetek pozytywnych odpowiedzi jest zazwyczaj bardzo niski (zbiór niezrównoważony), kluczowe było użycie stratyfikacji. Dzięki temu zagwarantowano, że w obu zbiorach znajdzie się dokładnie taki sam procent osób, które przyjęły kampanię, co zapobiega zjawisku trenowania modelu na niereprezentatywnej próbce.

In [12]:
def generate_sets():
    df = pd.read_csv('kampania.csv', sep='\t')

    print("Dane przed przygotowaniem:", df.shape)

    # uzupelnienie przychodu
    df['Income'] = df['Income'].fillna(df['Income'].median())

    # usuniecie stalych kolumn: Z_CostContact i Z_Revenue maja u wszystkich te same wartosci
    if 'Z_CostContact' in df.columns and 'Z_Revenue' in df.columns:
        df = df.drop(columns=['Z_CostContact', 'Z_Revenue'])

    # obczlienie wieku klienta
    current_year = 2026
    df['Age'] = current_year - df['Year_Birth']

    # usuniecie wartosci odstajacych (np. osob urodzonych w 1893 roku)
    df = df[df['Age'] < 100]

    # przeksztalcenie daty dolaczenia (Dt_Customer) na liczbowa: Liczba dni jako klient i zamiana tekstu na typ datetime
    df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
    # obliczenie roznicy w dniach (dni beda wzgledem najnowszej daty w bazie)
    latest_date = df['Dt_Customer'].max()
    df['Days_as_Customer'] = (latest_date - df['Dt_Customer']).dt.days

    # usuniecie pierwotnych kolumn
    df = df.drop(columns=['ID', 'Year_Birth', 'Dt_Customer'])

    # zamiana kolumn 'Education' i 'Marital_Status' na format bool'owy
    categorical_features = ['Education', 'Marital_Status']
    df = pd.get_dummies(df, columns=categorical_features, drop_first=True)

    bool_cols = df.select_dtypes(include='bool').columns
    df[bool_cols] = df[bool_cols].astype(int)

    # Oddzielenie cech (X) od zmiennej celu (y)
    X = df.drop(columns=['Response'])
    y = df['Response']

    # Podzial na zbior treningowy (80%) i testowy (20%)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print("Dane po przygotowaniu:", df.shape)
    print("Wymiary X_train:", X_train.shape)
    print("Wymiary X_test:", X_test.shape)

    return X_train, X_test, y_train, y_test



# Przypadek bazowy – Modele na oryginalnych danych ciągłych
W kroku drugim podjęto próbę wytrenowania modeli KNN (K-Nearest Neighbors) oraz SVM (Support Vector Machines) na danych, które przeszły jedynie obróbkę wstępną, opisaną powyzej. Najważniejszym aspektem tego eksperymentu jest fakt, że nie zastosowano żadnej techniki standaryzacji, skalowania ani dyskretyzacji danych ciągłych.

Algorytmy takie jak KNN i SVM opierają swoje działanie na mierzeniu odległości geometrycznej między punktami danych w przestrzeni wielowymiarowej.

Analiza wyników eksperymentu bazowego:

- Złudzenie wysokiej dokładności (Accuracy Paradox):
    Oba modele osiągnęły stosunkowo wysoką dokładność na poziomie około 84% (KNN) oraz 85% (SVM). W kontekście oceny kampanii marketingowych jest to jednak miara wysoce zwodnicza. Zbiór testowy jest asymetryczny – około 85% klientów w ogóle nie reaguje na promocje. W związku z tym, model, który po prostu "zgaduje", że żaden klient nie przyjmie oferty, automatycznie osiąga 85% poprawności.

- Brak zdolności predykcyjnych dla klasy pozytywnej:
    O rzeczywistej przydatności modelu świadczą metryki Recall (Czułość) oraz F1-Score dla mniejszościowej klasy 1 ("kupi").

    Model SVM wygenerował macierz błędów, w której wszystkie przewidywania padły na klasę 0. Czułość wyniosła absolutne 0.00. Model nie potrafił zidentyfikować ani jednego klienta podatnego na kampanię.

    Model KNN osiągnął Czułość na poziomie zaledwie ~9% i bardzo niski wskaźnik F1-Score, co oznacza, że większość potencjalnych kupujących została całkowicie zignorowana.

- Wnioski - "Klątwa skali":
    Powyższe, bardzo słabe wyniki są bezpośrednim dowodem na to, jak słabe jest uczenie modeli opartych na odległości danymi surowymi o różnych rzędach wielkości.
    W naszym zbiorze cecha Income przyjmuje wartości rzędu dziesiątek tysięcy (np. 60 000), podczas gdy cecha np. NumWebPurchases przyjmuje wartości od 0 do 10. Wzór na odległość euklidesową sprawia, że ogromna różnica w dochodach całkowicie "zagłusza" i marginalizuje wpływ mniejszych, ale waznych cech. Modele przesterowały się na jedną zmienną.

In [13]:
X_train, X_test, y_train, y_test = generate_sets()

knn_model = KNeighborsClassifier()
svm_model = SVC(kernel='rbf', random_state=42)

print("Trenowanie modelu KNN")
knn_model.fit(X_train, y_train)

print("Trenowanie modelu SVM")
svm_model.fit(X_train, y_train)

# wykonywanie predykcji na zbiorze testowym
knn_preds = knn_model.predict(X_test)
svm_preds = svm_model.predict(X_test)

def evaluate_model(model_name, y_true, y_pred):
    print(f"Wyniki dla modelu: {model_name}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision:{precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:   {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-Score: {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print("Macierz błędów:")
    print(confusion_matrix(y_true, y_pred))

evaluate_model("KNN", y_test, knn_preds)
evaluate_model("SVM", y_test, svm_preds)

Dane przed przygotowaniem: (2240, 29)
Dane po przygotowaniu: (2237, 35)
Wymiary X_train: (1789, 34)
Wymiary X_test: (448, 34)
Trenowanie modelu KNN
Trenowanie modelu SVM
Wyniki dla modelu: KNN
Accuracy: 0.8371
Precision:0.3333
Recall:   0.0896
F1-Score: 0.1412
Macierz błędów:
[[369  12]
 [ 61   6]]
Wyniki dla modelu: SVM
Accuracy: 0.8504
Precision:0.0000
Recall:   0.0000
F1-Score: 0.0000
Macierz błędów:
[[381   0]
 [ 67   0]]
